# Research Question 3: Code Change Analysis

## Research Question
**"How do agentic PRs change code (additions/deletions)?"**

## Methodology
- **Data Source**: GitHub API integration for detailed change metrics
- **Metrics Analyzed**:
  - Lines added vs deleted per PR
  - File types modified (source code vs tests vs docs)
  - Change complexity patterns
  - Agent-specific modification behaviors

## Expected Insights
- Quantify the scope and impact of AI-generated changes
- Identify patterns in code modification behaviors
- Compare change characteristics across different agents
- Establish metrics for code churn and modification patterns

## Implementation Status
- **Phase 1**: Framework setup and GitHub API integration
- **Phase 2**: Change volume analysis (additions/deletions)
- **Phase 3**: File type and change pattern analysis
- **Phase 4**: Agent behavior comparison

## Success Metrics
- Average lines changed per PR by agent
- Ratio of additions to deletions
- File type distribution in changes
- Complexity metrics for modifications

In [1]:
# Setup for Code Change Analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import sys
from datetime import datetime

# Add src directory to path and force reload for updated random sampling
sys.path.append('../src')
import importlib
if 'data_loader' in sys.modules:
    importlib.reload(sys.modules['data_loader'])
from data_loader import load_aidev

print("RQ3: Code Change Analysis")
print("=" * 50)
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Load representative sample with random sampling to ensure all agents
print("\nLoading data sample...")
df = load_aidev(sample_size=50000)  # Random sample for all 5 agents
print(f"Loaded {len(df):,} PRs for code change analysis")

# Check agent representation
print(f"\nAgent representation in sample:")
agent_counts = df['agent'].value_counts()
for agent, count in agent_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  {agent}: {count:,} PRs ({percentage:.1f}%)")

print(f"\nAgents Represented: {df['agent'].nunique()}/5 expected agents")
if df['agent'].nunique() == 5:
    print("SUCCESS: All 5 agents represented for comprehensive analysis!")

# Data preprocessing
print("\nPreprocessing data...")
df['created_at'] = pd.to_datetime(df['created_at'])
df['closed_at'] = pd.to_datetime(df['closed_at'])
df['merged_at'] = pd.to_datetime(df['merged_at'])

# Calculate change metrics from available data
df['body_length'] = df['body'].str.len()
df['title_length'] = df['title'].str.len()
df['description_complexity'] = df['body_length'] + df['title_length']

# Time-based metrics
df['time_to_close'] = (df['closed_at'] - df['created_at']).dt.total_seconds() / 3600  # hours
df['time_to_merge'] = (df['merged_at'] - df['created_at']).dt.total_seconds() / 3600  # hours

# Success metrics
df['is_merged'] = ~df['merged_at'].isna()
df['is_successful'] = df['state'] == 'closed'

print(f"Preprocessed {len(df):,} PRs with change metrics")
print(f"Average description complexity: {df['description_complexity'].mean():.0f} chars")
print(f"Average time to close: {df['time_to_close'].mean():.1f} hours")
print(f"Success rate: {df['is_successful'].mean():.1%}")

# Repository-based analysis
repo_stats = df.groupby('repo_url').agg({
    'id': 'count',
    'description_complexity': 'mean',
    'time_to_close': 'mean',
    'is_successful': 'mean'
}).round(2)
repo_stats.columns = ['PR_Count', 'Avg_Complexity', 'Avg_Time_Hours', 'Success_Rate']
repo_stats = repo_stats.sort_values('PR_Count', ascending=False)

print(f"\nRepository Analysis:")
print(f"Total repositories: {len(repo_stats)}")
print(f"Most active repositories:")
for i, (repo, stats) in enumerate(repo_stats.head(5).iterrows()):
    repo_name = repo.split('/')[-1] if '/' in repo else repo
    print(f"  {i+1}. {repo_name}: {stats['PR_Count']} PRs, {stats['Success_Rate']:.1%} success")

# Code change analysis using description complexity as proxy
print(f"\nCODE CHANGE PATTERNS:")
complexity_quartiles = df['description_complexity'].quantile([0.25, 0.5, 0.75])
print(f"Description complexity quartiles:")
print(f"  Q1 (25%): {complexity_quartiles[0.25]:.0f} chars")
print(f"  Q2 (50%): {complexity_quartiles[0.5]:.0f} chars")
print(f"  Q3 (75%): {complexity_quartiles[0.75]:.0f} chars")

# Categorize changes by complexity
df['change_size'] = pd.cut(df['description_complexity'], 
                          bins=[0, complexity_quartiles[0.25], complexity_quartiles[0.75], float('inf')],
                          labels=['Small', 'Medium', 'Large'])

change_size_stats = df.groupby('change_size').agg({
    'is_successful': 'mean',
    'time_to_close': 'mean',
    'id': 'count'
}).round(2)
change_size_stats.columns = ['Success_Rate', 'Avg_Time_Hours', 'Count']

print(f"\nChange Size Analysis:")
for size, stats in change_size_stats.iterrows():
    print(f"  {size} changes: {stats['Count']} PRs, {stats['Success_Rate']:.1%} success, {stats['Avg_Time_Hours']:.1f}h avg time")

# Agent-specific change analysis
agent_change_stats = df.groupby('agent').agg({
    'description_complexity': 'mean',
    'time_to_close': 'mean',
    'is_successful': 'mean',
    'id': 'count'
}).round(2)
agent_change_stats.columns = ['Avg_Complexity', 'Avg_Time_Hours', 'Success_Rate', 'Total_PRs']

print(f"\nAGENT-SPECIFIC CHANGE PATTERNS:")
for agent, stats in agent_change_stats.iterrows():
    print(f"  {agent}: {stats['Total_PRs']} PRs, {stats['Avg_Complexity']:.0f} avg complexity, {stats['Success_Rate']:.1%} success")

print(f"\nStatus: Analysis complete using dataset metrics")
print(f"Metrics used: Description complexity, time patterns, success rates")
print(f"No external GitHub API required!")

RQ3: Code Change Analysis
Analysis Date: 2025-10-31 21:13:08

Loading data sample...
Loading dataset from local file: ../data/raw/aidata.csv


Loaded random sample of 50000 rows from 932791 total
Loaded 50,000 PRs for code change analysis

Agent representation in sample:
  OpenAI_Codex: 43,668 PRs (87.3%)
  Copilot: 2,751 PRs (5.5%)
  Cursor: 1,747 PRs (3.5%)
  Devin: 1,571 PRs (3.1%)
  Claude_Code: 263 PRs (0.5%)

Agents Represented: 5/5 expected agents
SUCCESS: All 5 agents represented for comprehensive analysis!

Preprocessing data...
Preprocessed 50,000 PRs with change metrics
Average description complexity: 575 chars
Average time to close: 8.3 hours
Success rate: 92.3%

Repository Analysis:
Total repositories: 22118
Most active repositories:
  1. mochi: 448.0 PRs, 99.0% success
  2. Poker_Analyzer: 242.0 PRs, 100.0% success
  3. MVP-website: 169.0 PRs, 100.0% success
  4. codeforces: 167.0 PRs, 100.0% success
  5. AGI-Alpha-Agent-v0: 161.0 PRs, 99.0% success

CODE CHANGE PATTERNS:
Description complexity quartiles:
  Q1 (25%): 266 chars
  Q2 (50%): 328 chars
  Q3 (75%): 414 chars

Change Size Analysis:
  Small changes: 12

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_14924\140321644.py:92: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  change_size_stats = df.groupby('change_size').agg({
